# Import MPI Values

In [1]:
import pandas as pd
import os
mpi_df = pd.read_csv('/project/mclark/SCAMPI_datasets/IgG4/classify_by_dilation-sameTH/with-MPI/combined_SAM_scores_filtered5_10.csv')
valid_comps = [
    '115405_Area2','24052882340623_Area1','2122622117581_Area1'
]
mpi_df = mpi_df[mpi_df['CompName'].isin(valid_comps)]
mpi_df.shape

(82075, 295)

# Updated Processing Database

In [2]:
base_path = '/project/mclark/For/Bayard/Ground_truthing/IGG4/'
write_path = '/project/mclark/For/Bayard/Ground_truthing/IGG4/cut_samples-MPO/'

def read_file_as_set(filename):
    with open(filename, 'r') as file:
        return {os.path.basename(line.strip()) for line in file}

def write_set_to_file(filename, data_set):
    with open(filename, 'w') as file:
        for item in sorted(data_set):
            item = write_path + item
            file.write(item + '\n')

# File paths
processed_set = read_file_as_set(base_path + 'processed-MPO.txt')
skipped_set = read_file_as_set(base_path + 'hard_skip-MPO.txt')
labeled_set = read_file_as_set(base_path + 'labeled-MPO.txt')
to_process_file = base_path + 'to_process-MPO.txt'

# Read and filter the to_process set
to_process_set = read_file_as_set(to_process_file)
filtered_to_process_set = to_process_set - processed_set - skipped_set - labeled_set

# Write the updated set back to the file
write_set_to_file(to_process_file, filtered_to_process_set)

processed_file = base_path + 'processed-MPO.txt'
skipped_set = read_file_as_set(base_path + 'hard_skip-MPO.txt')
labeled_set = read_file_as_set(base_path + 'labeled-MPO.txt')

# Read and filter the processed set
processed_set = read_file_as_set(processed_file)
filtered_processed_set = processed_set - labeled_set - skipped_set

# Write the updated set back to the file
write_set_to_file(processed_file, filtered_processed_set)

processed_set = read_file_as_set(base_path + 'processed-MPO.txt')
skipped_set = read_file_as_set(base_path + 'hard_skip-MPO.txt')
to_process_set = read_file_as_set(base_path + 'to_process-MPO.txt')
labeled_set = read_file_as_set(base_path + 'labeled-MPO.txt')
print(len(processed_set))
print(len(skipped_set))
print(len(to_process_set))
print(len(labeled_set))

38
3050
3867
885


# Select approximate counts to grab per channel (set amounts higher then amount wanted, as the script will filter ones that have already been written)

In [3]:
pull = [ ('BDCA2',300)]

# Fix Cell_ID key here - not sure if Unnamed: 0 is consistent

In [4]:
lst = []
for (key,count) in pull:
    local_lst = []
    key = key + '_Int-mean_WC'
    per_comp = count // len(valid_comps)

    for comp in valid_comps:
        comp_df = mpi_df[mpi_df['CompName'] == comp].copy()
        local_df = comp_df.nlargest(int(len(comp_df) * 0.95), key)
        sample_size = min(count, len(local_df))
        local_df = local_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        for index, row in local_df.iterrows():
            sr = str(row['CompName']) + '_' + str(int(row['CellID'])) + '_r_' + str(int(row['GlobalMaskCentroidRow'])) + '.0_c_' + str(int(row['GlobalMaskCentroidCol'])) + '.0.npy'
            if sr not in processed_set and sr not in skipped_set and sr not in to_process_set and sr not in labeled_set:
                local_lst.append(sr)

    print(key, len(local_lst),"cells pulled")
    lst = lst + local_lst

BDCA2_Int-mean_WC 576 cells pulled


In [5]:
tp = set(list(to_process_set) + lst)
write_set_to_file(to_process_file, tp)